# 06 - Model: Cosine Similarity

# Imports and Load Data

In [1]:
import pandas as pd
import numpy as np
import json
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv('../data/processed/featured_dataset.csv')
df_full_standard = pd.read_csv('../data/processed/featured_full_standard.csv')
df_full_minmax = pd.read_csv('../data/processed/featured_full_minmax.csv')
df_audio_standard = pd.read_csv('../data/processed/featured_audio_standard.csv')
df_audio_minmax = pd.read_csv('../data/processed/featured_audio_minmax.csv')

with open('../config/feature_sets.json', 'r') as f:
    feature_sets = json.load(f)

audio_features = feature_sets['audio_features']
full_features = feature_sets['full_features']

print('shape:', df.shape)
print('audio features:', len(audio_features))
print('full features:', len(full_features))

shape: (88167, 32)
audio features: 11
full features: 21


# Cosine Similarity Function

In [2]:
def recommend_cosine(song_name, df, features, scaled_df, n=10):
    # find song index
    matches = df[df['track_name'].str.lower() == song_name.lower()]
    
    if matches.empty:
        print(f'Song "{song_name}" not found')
        return None
    
    idx = matches.index[0]
    song_features = scaled_df[features].iloc[idx].values.reshape(1, -1)
    
    # compute similarity
    similarity = cosine_similarity(song_features, scaled_df[features].values)[0]
    
    # get top n similar songs excluding itself
    similar_indices = similarity.argsort()[::-1][1:n+1]
    
    results = df.iloc[similar_indices][['track_name', 'artists', 'track_genre', 'popularity']].copy()
    results['similarity_score'] = similarity[similar_indices].round(4)
    results = results.reset_index(drop=True)
    results.index += 1
    
    return results

# Experiment 1 - Audio Features with StandardScaler

In [3]:
song = 'Shape of You'
print(f'Recommendations for: {song}')
print('Features: audio | Scaling: StandardScaler\n')
results = recommend_cosine(song, df, audio_features, df_audio_standard)
print(results)

Recommendations for: Shape of You
Features: audio | Scaling: StandardScaler

                                           track_name  \
1   Overture in the French Style, Op. 2, BWV 831 (...   
2                             Darktown Strutters Ball   
3                                      La Última Copa   
4                     Life Is Just a Bowl of Cherries   
5                                Yaramaz Ne Oldu Sana   
6                                        Leafy Greens   
7                                Zabytoe tango, Ch. 2   
8                                              マシュケナダ   
9                     Утомлённое солнце (Расставание)   
10  String Quartet No. 1 in A Minor, Op. 41 No. 1:...   

                                 artists        track_genre  popularity  \
1      Johann Sebastian Bach;Orion Weiss  classical, german           0   
2                             Crazy Otto         honky-tonk          11   
3                             Julio Sosa              tango          2

In [4]:
song_data = df[df['track_name'].str.lower() == 'shape of you']
print(song_data[['track_name', 'artists', 'track_genre', 'popularity'] + audio_features].to_string())

         track_name     artists track_genre  popularity  danceability  energy  mode  speechiness  acousticness  instrumentalness  liveness  valence   tempo  duration_min  explicit
39595  Shape Of You  Andrew Foy      guitar          24         0.561   0.284     0       0.0491         0.791              0.86    0.1070    0.825  96.893          3.67         0
66487  Shape of You  Ed Sheeran         pop          86         0.825   0.652     0       0.0802         0.581              0.00    0.0931    0.931  95.977          3.90         0


In [5]:
def recommend_cosine(song_name, df, features, scaled_df, artist=None, n=10):
    matches = df[df['track_name'].str.lower() == song_name.lower()]
    
    if artist:
        matches = matches[matches['artists'].str.lower().str.contains(artist.lower())]
    
    if matches.empty:
        print(f'Song "{song_name}" not found')
        return None
    
    idx = matches.index[0]
    print(f'Found: {df.loc[idx, "track_name"]} by {df.loc[idx, "artists"]}')
    print(f'Genre: {df.loc[idx, "track_genre"]}, Popularity: {df.loc[idx, "popularity"]}')
    print()
    
    song_features = scaled_df[features].iloc[idx].values.reshape(1, -1)
    similarity = cosine_similarity(song_features, scaled_df[features].values)[0]
    similar_indices = similarity.argsort()[::-1][1:n+1]
    
    results = df.iloc[similar_indices][['track_name', 'artists', 'track_genre', 'popularity']].copy()
    results['similarity_score'] = similarity[similar_indices].round(4)
    results = results.reset_index(drop=True)
    results.index += 1
    
    return results

In [6]:
results = recommend_cosine('Shape of You', df, audio_features, df_audio_standard, artist='Ed Sheeran')
print(results)

Found: Shape of You by Ed Sheeran
Genre: pop, Popularity: 86

                  track_name                                     artists  \
1                      Yaaro                           Santesh;Amos Paul   
2           No Hay Carretera                              La Misma Gente   
3             Kara Saplantım                                     Kayahan   
4   Copacabana (At the Copa)                               Barry Manilow   
5                        Low                          Larry Gaaga;Wizkid   
6             Malligai Poove                        Sujatha;Unnikrishnan   
7                Yolla Yarim                                 Barış Manço   
8    No Me Vuelvo a Enamorar                                    Los Apus   
9        Walking on Sunshine  Pickin' On Series;Brad Davis;Lou Ann Price   
10              Azúcar Negra                                  Celia Cruz   

   track_genre  popularity  similarity_score  
1        malay          29            0.9815  
2      

# Experiment 2 - Full Features with StandardScaler

In [7]:
results = recommend_cosine('Shape of You', df, full_features, df_full_standard, artist='Ed Sheeran')
print(results)

Found: Shape of You by Ed Sheeran
Genre: pop, Popularity: 86

                    track_name               artists track_genre  popularity  \
1     Copacabana (At the Copa)         Barry Manilow       disco          56   
2          El aire de la calle      Los Delinquentes     spanish          61   
3                 Mad Over You               Runtown   dancehall          58   
4                     Unlonely            Jason Mraz    acoustic          51   
5         The Ants Go Marching    Super Simple Songs    children          54   
6   The Richest Man In Babylon  Thievery Corporation    trip-hop          54   
7                        Fever                Wizkid   dancehall          56   
8                   On the Low             Burna Boy   dancehall          74   
9                       _WORLD             SEVENTEEN       k-pop          78   
10      La Vida Es Un Carnaval            Celia Cruz       salsa          69   

    similarity_score  
1             0.9667  
2          

# Experiment 3 - Full Features with MinMaxScaler

In [8]:
results = recommend_cosine('Shape of You', df, full_features, df_full_minmax, artist='Ed Sheeran')
print(results)

Found: Shape of You by Ed Sheeran
Genre: pop, Popularity: 86

                                 track_name  \
1                       El aire de la calle   
2                  Copacabana (At the Copa)   
3                                       Low   
4                                  Vermedin   
5                                 Roll Deep   
6                             Give a Little   
7      Thenmozhi (From "Thiruchitrambalam")   
8   Showed Me (How I Fell In Love With You)   
9                              Mad Over You   
10                     The Ants Go Marching   

                                   artists    track_genre  popularity  \
1                         Los Delinquentes        spanish          61   
2                            Barry Manilow          disco          56   
3                       Larry Gaaga;Wizkid      dancehall          54   
4                           Umut Timur;MRC        turkish          56   
5                  Tegi Pannu;Manni Sandhu        hip-h

# Experiment 4 - Audio Features with MinMaxScaler

In [9]:
results = recommend_cosine('Shape of You', df, audio_features, df_audio_minmax, artist='Ed Sheeran')
print(results)

Found: Shape of You by Ed Sheeran
Genre: pop, Popularity: 86

                                    track_name  \
1                               Kara Saplantım   
2                          El aire de la calle   
3                                     Maranura   
4                             Passada Discreta   
5                          Walking on Sunshine   
6                               Mal Acostumado   
7   Du brukade kalla mig för baby (feat. Kaah)   
8                            As - Luxury Remix   
9                                        Yaaro   
10                                 Yolla Yarim   

                                       artists track_genre  popularity  \
1                                      Kayahan     turkish          39   
2                             Los Delinquentes     spanish          61   
3                  William Luna;Julio Humala L      guitar          25   
4                                     MC Marks        funk          46   
5   Pickin' On Se

# Experiment 5 - Genre Filtered Cosine Similarity

In [10]:
def recommend_cosine_genre_filtered(song_name, df, features, scaled_df, artist=None, n=10):
    matches = df[df['track_name'].str.lower() == song_name.lower()]
    
    if artist:
        matches = matches[matches['artists'].str.lower().str.contains(artist.lower())]
    
    if matches.empty:
        print(f'Song "{song_name}" not found')
        return None
    
    idx = matches.index[0]
    song_genre = df.loc[idx, 'track_genre']
    print(f'Found: {df.loc[idx, "track_name"]} by {df.loc[idx, "artists"]}')
    print(f'Genre: {song_genre}, Popularity: {df.loc[idx, "popularity"]}')
    print()

    # filter by overlapping genre
    def has_common_genre(genre_str):
        song_genres = set(song_genre.split(', '))
        other_genres = set(genre_str.split(', '))
        return bool(song_genres & other_genres)

    genre_mask = df['track_genre'].apply(has_common_genre)
    filtered_df = df[genre_mask]
    filtered_scaled = scaled_df[genre_mask]

    song_features = scaled_df[features].iloc[idx].values.reshape(1, -1)
    similarity = cosine_similarity(song_features, filtered_scaled[features].values)[0]

    similar_indices = similarity.argsort()[::-1][1:n+1]
    actual_indices = filtered_df.index[similar_indices]

    results = df.loc[actual_indices][['track_name', 'artists', 'track_genre', 'popularity']].copy()
    results['similarity_score'] = similarity[similar_indices].round(4)
    results = results.reset_index(drop=True)
    results.index += 1

    return results

results = recommend_cosine_genre_filtered('Shape of You', df, full_features, df_full_standard, artist='Ed Sheeran')
print(results)

Found: Shape of You by Ed Sheeran
Genre: pop, Popularity: 86

                              track_name  \
1                                   Naah   
2        There's Nothing Holdin' Me Back   
3                          Bijlee Bijlee   
4                              Attention   
5                            Summer High   
6        Ghalat Fehmi - From "Superstar"   
7                         Mitti De Tibbe   
8                            Kya Baat Ay   
9                     Kya Mujhe Pyar Hai   
10  Thenmozhi (From "Thiruchitrambalam")   

                                   artists    track_genre  popularity  \
1                            Harrdy Sandhu            pop          63   
2                             Shawn Mendes     dance, pop          86   
3                            Harrdy Sandhu            pop          74   
4                             Charlie Puth     dance, pop          83   
5                               AP Dhillon   hip-hop, pop          83   
6          Asim

# Experiment 5 Verification - Multiple Songs

In [11]:
test_songs = [
    ('Blinding Lights', 'The Weeknd'),
    ('Bohemian Rhapsody', 'Queen'),
    ('Bad Guy', 'Billie Eilish'),
]

for song, artist in test_songs:
    print(f'\n{"="*50}')
    results = recommend_cosine_genre_filtered(song, df, full_features, df_full_standard, artist=artist)
    if results is not None:
        print(results)


Found: Blinding Lights by The Weeknd
Genre: pop, Popularity: 91

                        track_name           artists track_genre  popularity  \
1                      Unstoppable               Sia  dance, pop          81   
2                   Wildest Dreams      Taylor Swift         pop          80   
3              MIDDLE OF THE NIGHT        Elley Duhé         pop          90   
4                            Ghost     Justin Bieber         pop          88   
5                      Sufna Banke             Harvi         pop          66   
6                       Love Story      Taylor Swift         pop          77   
7           Orasaadha - Madras Gig    Vivek - Mervin         pop          67   
8   Deewane Hum Nahi - Version 1.0      Aditya Yadav         pop          64   
9                 Story of My Life     One Direction         pop          83   
10                           Yaari  Nikk;Avneet Kaur         pop          67   

    similarity_score  
1             0.9214  
2      

# Experiment 6 - Genre Filtered with MinMaxScaler

In [12]:
print('MinMaxScaler results:')
results = recommend_cosine_genre_filtered('Shape of You', df, full_features, df_full_minmax, artist='Ed Sheeran')
print(results)

MinMaxScaler results:
Found: Shape of You by Ed Sheeran
Genre: pop, Popularity: 86

                                        track_name  \
1             Thenmozhi (From "Thiruchitrambalam")   
2                  There's Nothing Holdin' Me Back   
3   Mayakkama Kalakkama (From "Thiruchitrambalam")   
4                                         Schedule   
5                  Ghalat Fehmi - From "Superstar"   
6                                         Saiyaara   
7                        Naah Goriye (From "Bala")   
8                                             Naah   
9                                    Jug Jug Jeeve   
10                                    Daru Badnaam   

                                              artists           track_genre  \
1              Santhosh Narayanan;Anirudh Ravichander         pop, pop-film   
2                                        Shawn Mendes            dance, pop   
3                         Dhanush;Anirudh Ravichander  k-pop, pop, pop-film   
4    

# Experiment 7 - Genre Filtered Audio Features StandardScaler

In [13]:
results = recommend_cosine_genre_filtered('Shape of You', df, audio_features, df_audio_standard, artist='Ed Sheeran')
print(results)

Found: Shape of You by Ed Sheeran
Genre: pop, Popularity: 86

                                        track_name  \
1                        Naah Goriye (From "Bala")   
2                                             Naah   
3                  There's Nothing Holdin' Me Back   
4                                      Kya Baat Ay   
5                                         Saiyaara   
6                           Yaad Piya Ki Aane Lagi   
7     Chola Chola (From "Ponniyin Selvan Part -1")   
8   Mayakkama Kalakkama (From "Thiruchitrambalam")   
9                                        Attention   
10                                        Schedule   

                                              artists           track_genre  \
1                  B Praak;Harrdy Sandhu;Swasti Mehul                   pop   
2                                       Harrdy Sandhu                   pop   
3                                        Shawn Mendes            dance, pop   
4                          

# Save Experiment Results

In [14]:
import os
os.makedirs('../reports/tables', exist_ok=True)

results_summary = {
    'experiment': [1, 2, 3, 4, 5, 6, 7],
    'features': ['audio', 'full', 'full', 'audio', 'full', 'full', 'audio'],
    'scaling': ['standard', 'standard', 'minmax', 'minmax', 'standard', 'minmax', 'standard'],
    'genre_filter': [False, False, False, False, True, True, True],
    'result_quality': ['poor', 'poor', 'poor', 'poor', 'good', 'good', 'good'],
    'notes': [
        'wrong genres, low popularity',
        'wrong genres, slightly better popularity',
        'wrong genres, inflated scores',
        'wrong genres, very high scores',
        'correct genres, diverse artists, best overall',
        'correct genres, biased toward similar feature ranges',
        'correct genres, good results'
    ]
}

results_df = pd.DataFrame(results_summary)
results_df.to_csv('../reports/tables/cosine_experiment_results.csv', index=False)
print(results_df.to_string())

   experiment features   scaling  genre_filter result_quality                                                 notes
0           1    audio  standard         False           poor                          wrong genres, low popularity
1           2     full  standard         False           poor              wrong genres, slightly better popularity
2           3     full    minmax         False           poor                         wrong genres, inflated scores
3           4    audio    minmax         False           poor                        wrong genres, very high scores
4           5     full  standard          True           good         correct genres, diverse artists, best overall
5           6     full    minmax          True           good  correct genres, biased toward similar feature ranges
6           7    audio  standard          True           good                          correct genres, good results


# Save Best Model

In [15]:
import pickle

# save best configuration
best_config = {
    'model': 'cosine_similarity',
    'features': 'full_features',
    'scaling': 'StandardScaler',
    'genre_filter': True,
}

# save feature matrix for best config
feature_matrix = df_full_standard[full_features].values

os.makedirs('../models/content_based', exist_ok=True)

# save feature matrix
np.save('../models/content_based/cosine_feature_matrix.npy', feature_matrix)

# save config
with open('../models/content_based/cosine_config.json', 'w') as f:
    json.dump(best_config, f, indent=4)

# save dataframe index mapping
df[['track_id', 'track_name', 'artists', 'track_genre', 'popularity']].to_csv(
    '../models/content_based/cosine_song_index.csv', index=True)

print('saved files:')
print('1. models/content_based/cosine_feature_matrix.npy')
print('2. models/content_based/cosine_config.json')
print('3. models/content_based/cosine_song_index.csv')

saved files:
1. models/content_based/cosine_feature_matrix.npy
2. models/content_based/cosine_config.json
3. models/content_based/cosine_song_index.csv


# Conclusion

In [16]:
print('='*50)
print('COSINE SIMILARITY MODEL - CONCLUSION')
print('='*50)

print('\n--- Experiments Summary ---')
print('Exp 1: audio + standard (no filter)   → failed: wrong genres')
print('Exp 2: full + standard (no filter)    → failed: wrong genres')
print('Exp 3: full + minmax (no filter)      → failed: wrong genres')
print('Exp 4: audio + minmax (no filter)     → failed: wrong genres')
print('Exp 5: full + standard + genre filter → best: correct genres, diverse')
print('Exp 6: full + minmax + genre filter   → good: correct but biased')
print('Exp 7: audio + standard + genre filter→ good: correct genres')

print('\n--- Best Configuration ---')
print('Features     → full_features (21)')
print('Scaling      → StandardScaler')
print('Genre Filter → Yes')

print('\n--- Why Best Configuration Works ---')
print('full_features → richer song representation')
print('StandardScaler → honest scores, diverse results')
print('genre filter  → essential for musical relevance')

print('\n--- Key Findings ---')
print('1. genre filter is essential for cosine similarity')
print('2. without genre filter → mathematically similar but musically irrelevant')
print('3. full features outperform audio features alone')
print('4. StandardScaler better than MinMaxScaler for diversity')
print('5. MinMaxScaler inflates scores and biases recommendations')

print('\n--- Limitations ---')
print('1. recommendations limited to same genre only')
print('2. cannot discover cross genre songs')
print('3. computationally expensive for large datasets')

print('\n--- Best Use Case ---')
print('similar song recommendation within same genre')
print('works well for: find me songs like Shape of You')

print('\n--- Saved Files ---')
print('models/content_based/cosine_feature_matrix.npy')
print('models/content_based/cosine_config.json')
print('models/content_based/cosine_song_index.csv')
print('reports/tables/cosine_experiment_results.csv')

COSINE SIMILARITY MODEL - CONCLUSION

--- Experiments Summary ---
Exp 1: audio + standard (no filter)   → failed: wrong genres
Exp 2: full + standard (no filter)    → failed: wrong genres
Exp 3: full + minmax (no filter)      → failed: wrong genres
Exp 4: audio + minmax (no filter)     → failed: wrong genres
Exp 5: full + standard + genre filter → best: correct genres, diverse
Exp 6: full + minmax + genre filter   → good: correct but biased
Exp 7: audio + standard + genre filter→ good: correct genres

--- Best Configuration ---
Features     → full_features (21)
Scaling      → StandardScaler
Genre Filter → Yes

--- Why Best Configuration Works ---
full_features → richer song representation
StandardScaler → honest scores, diverse results
genre filter  → essential for musical relevance

--- Key Findings ---
1. genre filter is essential for cosine similarity
2. without genre filter → mathematically similar but musically irrelevant
3. full features outperform audio features alone
4. Standar